# Choose repository files

Choose a local folder, see candidate files, and adjust which folders or file types are selected. This preview uses file names and sizes only: it does not read file contents, run repository code, load a model, or create an index.

Run setup first and select QAChat's `.venv` kernel. No downloaded LLM is needed. Run one cell at a time with Shift+Enter.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "app_core/repository.py").is_file():
    root = root.parent
assert (root / "app_core/repository.py").is_file(), "Open this notebook from QAChat."
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
assert Path(sys.prefix).resolve() == (root / ".venv").resolve(), "Select QAChat's .venv kernel."

from app_core.repository import scan_repository

print("Python:", sys.executable)

## 1. Choose a local project folder

Start with QAChat itself. For another project, replace `root` with `Path("/your/project/folder")`. A normal folder works; Git is not required. Relative paths start at QAChat. Use a specific project folder, not your entire home folder or drive.

This does not clone GitHub repositories or copy your files. Change the folder, then rerun all cells below.

In [ ]:
REPOSITORY = root
# Example: REPOSITORY = Path("~/my-project")

report = scan_repository(REPOSITORY)
print("Repository:", report["repository"])
print("Default file types:", ", ".join(report["extensions"]))
print("Candidate files:", len(report["selected_files"]))

## 2. See the initial candidates

The preview lists common code, text, configuration, and notebook file extensions. `selected` means the file matches our name and size rules; it does not yet mean its contents are valid text or supported by a document reader.

To keep output short, show the first 40 files. The complete list remains in `report["selected_files"]`.

In [ ]:
for name in report["selected_files"][:40]:
    print(name)
print(f"Showing {min(40, len(report['selected_files']))} of {len(report['selected_files'])} candidate files.")

## 3. Adjust the selection

Choose extensions such as `.py`, `.md`, or `.m`. An empty `INCLUDE_PATHS` means all otherwise eligible files. Enter a folder or filename to narrow it, for example `["app_core"]` or `["README.md"]`. An empty `FILE_TYPES` selects no files.

These paths start inside the chosen repository. Use the spelling shown above and `/` between folders. No wildcards are supported. Exclusions always win. A folder entry applies to its children too.

The default example selects Python and Markdown but excludes `tests`. Change these values and rerun the remaining cells. The size limit is in bytes: 1,048,576 bytes is 1 MiB.

In [ ]:
FILE_TYPES = [".py", ".md"]
INCLUDE_PATHS = []
EXCLUDE_PATHS = ["tests"]
MAX_FILE_BYTES = 1_048_576

## 4. Preview your selection

This calls the same function the future application will use. No files are changed. If `complete` is false, check the reported errors before relying on the selection. A scan stops with an error after 10,000 visited entries; choose a smaller folder or explicitly raise `max_entries` in the call.

In [ ]:
report = scan_repository(
    REPOSITORY,
    extensions=FILE_TYPES,
    include_paths=INCLUDE_PATHS,
    exclude_paths=EXCLUDE_PATHS,
    max_file_bytes=MAX_FILE_BYTES,
)

selected = [entry for entry in report["entries"] if entry["status"] == "selected"]
for entry in selected[:40]:
    print(f"{entry['path']} ({entry['size_bytes']:,} bytes)")
print(f"Showing {min(40, len(selected))} of {len(selected)} selected files.")
if not selected:
    print("No files match. Check the folder, file types, and include/exclude paths.")

## 5. Understand what was skipped

Hidden entries, common model/environment/generated folders, known weight/key files, and links are always skipped. These safeguards cannot be overridden by `INCLUDE_PATHS`. Skipped folders appear once; their children are not visited or counted individually.

This preview does **not** read `.gitignore`. Git's rules decide what can be committed; our selection decides what is a candidate for later reading. Add your own private or generated folders to `EXCLUDE_PATHS`, especially if they have custom names. Name-based rules are not a secret scanner.

Nothing checks the bytes inside a file yet. Reading and validating contents will come with document chunking.

In [ ]:
skipped = [entry for entry in report["entries"] if entry["status"] == "skipped"]
for entry in skipped[:40]:
    suffix = "/" if entry["kind"] == "folder" else ""
    print(f"{entry['path']}{suffix} — {entry['reason']}")
print(f"Showing {min(40, len(skipped))} of {len(skipped)} skipped entries.")
print("Scan complete:", report["complete"])

## 6. Check the result

These checks use the actual returned sizes and status. The result is an in-memory selection report, not saved text or a search index. You can try another selection without repeating setup or downloading models.

The notebook itself may save displayed paths in its outputs. Clear outputs before committing or sharing it.

In [ ]:
assert report["complete"], "Some entries could not be inspected. Read the skip reasons above."
assert all(0 < entry["size_bytes"] <= MAX_FILE_BYTES for entry in selected)
{
    "repository": report["repository"],
    "selected_files": len(report["selected_files"]),
    "selected_bytes": report["selected_bytes"],
    "skipped_entries": len(skipped),
}

Next planned example: turn a few sentences into searchable numbers with an embedding model. Later, document chunking will read and validate the files selected here. Neither feature is implemented by this preview.